In [1]:
pip install lightgbm


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("crop_yield_final_cleaned.csv")

X = df.drop(columns=["Yield_tons_per_hectare"])
y = df["Yield_tons_per_hectare"]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


In [7]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [8]:
results = {}
lgbm_model.fit(X_train, y_train)
y_pred_lgbm = lgbm_model.predict(X_test)

mae, rmse, r2 = evaluate_model(y_test, y_pred_lgbm)

results["LightGBM"] = {
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032396 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 635
[LightGBM] [Info] Number of data points in the train set: 799815, number of used features: 20
[LightGBM] [Info] Start training from score 4.650228


In [9]:
final_results = pd.DataFrame(results).T.round(4)
final_results.sort_values("RMSE")


,MAE,RMSE,R2
LightGBM,0.3986,0.4996,0.9131


In [10]:
joblib.dump(lgbm_model, "lightgbm.pkl")

['lightgbm.pkl']